# Himawari → Singapore crops (2024–now, 10-min, daytime)

Streams raw Himawari-9 scans from NOAA's public AWS bucket and crops Singapore on Colab.
Each finished **month is zipped and downloaded to your computer** (browser Downloads folder),
then deleted from Colab. Only a tiny progress file is kept on Drive.

**Resumable:** if Colab disconnects, *Runtime → Run all* again — months already downloaded
are skipped; the interrupted month restarts.

Per scan: `sg_YYYYMMDD_HHMM.npz` (B03 reflectance + B13 brightness temp, 448×448) and
`himawari_sg_YYYYMMDD_HHMM00.png` (RGB composite the model reads). ≈1.4 GB per month zipped.

Tip: in Chrome settings turn off "Ask where to save each file" so downloads don't prompt.
Unzip everything into the project's `data/satellite_aws/` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK_ROOT = '/content/himawari_sg'                               # Colab scratch disk
DONE_FILE = '/content/drive/MyDrive/himawari_sg_done_months.txt'  # progress marker (tiny)
START     = '2024-01'          # first month to process (YYYY-MM)
END       = None               # None = up to the last complete month

In [ ]:
!pip -q install satpy pyresample s3fs

Get `himawari_aws.py`: upload it to `MyDrive/` once (or it is cloned from GitHub if pushed).

In [ ]:
import os, shutil, sys
DRIVE_COPY = '/content/drive/MyDrive/himawari_aws.py'
if os.path.exists(DRIVE_COPY):
    shutil.copy(DRIVE_COPY, '/content/himawari_aws.py')
elif not os.path.exists('/content/himawari_aws.py'):
    !git clone -q https://github.com/Qasim1507/Solar-Prediction.git /content/Project
    shutil.copy('/content/Project/himawari_aws.py', '/content/himawari_aws.py')
sys.path.insert(0, '/content')
import himawari_aws
print('loaded', himawari_aws.__file__)

## Quick test — one scan
Check the image shows Singapore / the Malay Peninsula and note the time per scan.

In [ ]:
from datetime import datetime
from PIL import Image
import numpy as np, time
t0 = time.time()
status, path = himawari_aws.extract_scan(datetime(2024, 2, 20, 1, 0), '/content/test_scan')
print(status, path, f'{time.time()-t0:.1f}s')
b = np.load(path.replace('himawari_sg_', 'sg_').replace('00.png', '.npz'))['bands'].astype('float32')
print('reflectance', np.nanmin(b[0]), np.nanmax(b[0]), '| BT K', np.nanmin(b[1]), np.nanmax(b[1]))
Image.open(path)

## Full run
One month at a time: extract → zip → download to your computer → mark done → delete from Colab.
Progress prints every 20 scans; missing scans are listed in each month's `missing.csv`.

In [ ]:
from datetime import datetime, timedelta
from google.colab import files

def done_months():
    return set(open(DONE_FILE).read().split()) if os.path.exists(DONE_FILE) else set()

start = datetime.strptime(START, '%Y-%m')
last  = datetime.strptime(END, '%Y-%m') if END else datetime.utcnow().replace(day=1) - timedelta(days=1)
m = start
while m <= last:
    tag = m.strftime('%Y-%m')
    nxt = (m.replace(day=28) + timedelta(days=4)).replace(day=1)
    if tag in done_months():
        print(tag, 'already downloaded — skipping')
        m = nxt
        continue
    month_root = os.path.join(WORK_ROOT, tag)
    counts = himawari_aws.run_range(m, nxt - timedelta(minutes=1), month_root,
                                    workers=os.cpu_count())
    zip_path = shutil.make_archive(f'/content/himawari_sg_{tag}', 'zip', month_root)
    print(tag, counts, f'{os.path.getsize(zip_path)/1e9:.2f} GB — downloading')
    files.download(zip_path)
    time.sleep(60)   # give the browser time to pull the file before deleting it
    with open(DONE_FILE, 'a') as f:
        f.write(tag + '\n')
    shutil.rmtree(month_root)
    m = nxt

If a download fails in the browser, remove that month from `himawari_sg_done_months.txt`
on Drive and rerun.